# Surrogate-Based Null Hypothesis Testing for RQA-ML Classification & Clustering

**Purpose:** Generate all data and publication-quality figures for the updated methods paper.

This notebook demonstrates that the nonlinear dynamical structure captured by Recurrence Quantification Analysis (RQA) features drives classification and clustering performance *beyond* what linear autocorrelation, amplitude distribution, or periodic structure alone can explain.

**Pipeline overview:**
1. Generate chaotic dynamical systems (Rossler, Lorenz, Henon) as ground-truth signals
2. Extract RQA features via `RQA2_ml.build_feature_table`
3. Run nested cross-validation on real features
4. Run surrogate-based null benchmarks (FT, AAFT, IAAFT) — each destroys a different aspect of nonlinear structure
5. Run label-permutation null for comparison
6. Apply Benjamini-Hochberg FDR correction
7. Repeat for unsupervised clustering
8. Generate publication figures

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Publication-quality defaults
matplotlib.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.figsize": (7, 4),
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.1,
})

from SMdRQA.RQA2 import RQA2, RQA2_simulators, RQA2_tests, RQA2_ml

print("Imports OK")

In [ ]:
# Output directory for figures
FIG_DIR = os.path.join(os.getcwd(), "..", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

# Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

print(f"Figures will be saved to: {os.path.abspath(FIG_DIR)}")

## 1. Generate Chaotic Time Series

We create three classes of signals from well-known chaotic dynamical systems:
- **Rossler attractor** (class 0)
- **Lorenz attractor** (class 1)
- **Henon map** (class 2)

Multiple realisations per class are generated with different initial conditions (via seeds) to simulate inter-subject variability, as would be typical in a neurophysiological or ecological study.

In [ ]:
N_PER_CLASS = 10       # realisations per dynamical system
SIG_LEN = 2000         # time-series length (points)

signals = []
labels = []

# --- Rossler (class 0) ---
for i in range(N_PER_CLASS):
    sim = RQA2_simulators(seed=SEED + i)
    x, _, _ = sim.rossler(tmax=500, n=SIG_LEN)
    # Add small observational noise
    x = x + 0.05 * rng.standard_normal(len(x))
    signals.append(x.astype(float))
    labels.append("rossler")

# --- Lorenz (class 1) ---
for i in range(N_PER_CLASS):
    sim = RQA2_simulators(seed=SEED + 100 + i)
    x, _, _ = sim.lorenz(tmax=50, n=SIG_LEN)
    x = x + 0.05 * rng.standard_normal(len(x))
    signals.append(x.astype(float))
    labels.append("lorenz")

# --- Henon (class 2) ---
for i in range(N_PER_CLASS):
    sim = RQA2_simulators(seed=SEED + 200 + i)
    x, _ = sim.henon(n=SIG_LEN)
    x = x + 0.01 * rng.standard_normal(len(x))
    signals.append(x.astype(float))
    labels.append("henon")

labels = np.array(labels)
print(f"Generated {len(signals)} signals: {np.unique(labels, return_counts=True)}")

### Figure 1: Example Time Series

Visualise one realisation from each class to illustrate the qualitative differences between the three chaotic systems.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(7, 5), sharex=True)
class_names = ["rossler", "lorenz", "henon"]
colors = ["#4C72B0", "#DD8452", "#55A868"]
example_idx = [0, N_PER_CLASS, 2 * N_PER_CLASS]  # first signal of each class

for ax, idx, name, color in zip(axes, example_idx, class_names, colors):
    sig = signals[idx][:500]  # show first 500 points for clarity
    ax.plot(sig, color=color, linewidth=0.6)
    ax.set_ylabel(name.capitalize())
    ax.set_xlim(0, 500)
    sns.despine(ax=ax)

axes[-1].set_xlabel("Time (samples)")
fig.suptitle("Figure 1: Example Time Series from Three Chaotic Systems", fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig1_example_timeseries.pdf"))
fig.savefig(os.path.join(FIG_DIR, "fig1_example_timeseries.png"))
plt.show()

## 2. Build RQA Feature Table

Extract whole-signal RQA measures and windowed summary statistics. This produces one feature vector per signal, suitable for ML classification and clustering.

In [ ]:
ml = RQA2_ml(rqa_kwargs={"normalize": True})

WINDOW_SIZE = 100
WINDOW_STEP = 20

features = ml.build_feature_table(
    signals, labels=labels,
    window_size=WINDOW_SIZE,
    window_step=WINDOW_STEP,
    window_stats=("mean", "median", "mode"),
    include_params=True,
)

feature_cols = [c for c in features.columns if c not in ("id", "label")]
X = features[feature_cols].values
y = features["label"].values

print(f"Feature table: {features.shape[0]} samples x {len(feature_cols)} features")
features.head()

## 3. Supervised Benchmark: Real Data (Nested CV)

Run the full nested cross-validation with feature selection on the real RQA features. This establishes the *real* classification performance that we will compare against the surrogate null distributions.

In [ ]:
MODELS = ("knn", "svm", "rf")

# Quick benchmark to select best model
results_df, best_model = ml.supervised_benchmark(
    X, y, models=MODELS, cv=5, scaler=True, random_state=SEED
)
print("Quick benchmark (5-fold CV):")
print(results_df.to_string(index=False))
print(f"\nBest model: {results_df.iloc[0]['model']}")

In [ ]:
# Full nested CV with feature selection (paper methodology)
BEST_MODEL = results_df.iloc[0]["model"]

real_results = ml.nested_cv_benchmark(
    X, y,
    model=BEST_MODEL,
    outer_iterations=100,
    test_fraction=1.0 / 3,
    inner_splits=2,
    inner_iterations=10,
    feature_selection="auto",
    scaler=True,
    random_state=SEED,
)

print(f"Nested CV ({BEST_MODEL}): "
      f"Accuracy = {np.mean(real_results['accuracy']):.3f} "
      f"({np.std(real_results['accuracy']):.3f}), "
      f"ROC AUC = {np.nanmean(real_results['roc_auc']):.3f} "
      f"({np.nanstd(real_results['roc_auc']):.3f})")
print(f"\nMost frequently selected features:")
print(real_results["feature_frequency"].head(10))

## 4. Surrogate-Based Null Hypothesis Testing (Supervised)

This is the core methodological contribution. We test three surrogate types, each encoding a different null hypothesis:

| Surrogate | What it preserves | Null hypothesis |
|-----------|-------------------|-----------------|
| **FT** | Power spectrum | Linear autocorrelation alone explains classification |
| **AAFT** | Spectrum + amplitude distribution | Linear structure + marginal distribution suffice |
| **IAAFT** | Refined spectrum + amplitude | Tighter linear null |

For each surrogate type, we:
1. Replace every signal with its surrogate
2. Re-extract RQA features from the surrogate signals
3. Run the same nested CV pipeline
4. Repeat `n_surrogate_iterations` times to build a null distribution
5. Compare real performance vs null using rank-based p-values

Additionally, we include a standard label-permutation baseline for comparison.

**Key insight:** If real accuracy significantly exceeds the surrogate null, the *nonlinear* structure (not just linear properties) is what drives classification.

In [ ]:
%%time

# ------------------------------------------------------------------
# NOTE: This is the most computationally expensive cell.
#
# With the default settings below (n_surrogate_iterations=20,
# surrogate_outer_iterations=30), expect ~15-30 min depending on
# hardware.  For a quick test run, reduce n_surrogate_iterations
# to 3 and surrogate_outer_iterations to 5.
# ------------------------------------------------------------------

N_SURR_ITER = 20           # surrogate datasets per algorithm
SURR_OUTER_ITER = 30       # nested CV iterations per surrogate dataset
N_PERMUTATIONS = 100       # label-permutation iterations

surr_results = ml.surrogate_null_benchmark(
    signals, labels,
    window_size=WINDOW_SIZE,
    window_step=WINDOW_STEP,
    window_stats=("mean", "median", "mode"),
    include_params=True,
    surrogate_kinds=("FT", "AAFT", "IAAFT"),
    n_surrogate_iterations=N_SURR_ITER,
    model=BEST_MODEL,
    outer_iterations=100,
    surrogate_outer_iterations=SURR_OUTER_ITER,
    inner_splits=2,
    inner_iterations=10,
    feature_selection="auto",
    scaler=True,
    random_state=SEED,
    alpha=0.05,
    correction="fdr_bh",
    include_permutation=True,
    n_permutations=N_PERMUTATIONS,
    verbose=True,
)

print("\nDone.")

### Table 1: Surrogate Null Benchmark Summary

This table is the core results table for the paper. It shows real vs null performance for each surrogate type, with FDR-corrected p-values and Cohen's d effect sizes.

In [ ]:
summary = surr_results["summary"]

# Display key columns in a clean format
display_cols = [
    "null_type", "null_hypothesis",
    "real_accuracy", "null_mean_accuracy", "null_std_accuracy",
    "p_value_accuracy", "adjusted_p_accuracy", "significant_accuracy",
    "effect_size_accuracy",
]
table1 = summary[display_cols].copy()
table1.columns = [
    "Null Type", "Null Hypothesis",
    "Real Acc.", "Null Mean Acc.", "Null Std Acc.",
    "p (raw)", "p (FDR)", "Significant", "Cohen's d",
]
# Round for display
for col in table1.select_dtypes(include="number").columns:
    table1[col] = table1[col].map(lambda x: f"{x:.4f}" if abs(x) < 100 else f"{x:.1f}")
table1

In [ ]:
# Save full summary to CSV for the paper's supplementary materials
summary.to_csv(os.path.join(FIG_DIR, "table1_surrogate_null_supervised.csv"), index=False)
print("Saved table1_surrogate_null_supervised.csv")

### Figure 2: Surrogate Null Distributions vs Real Performance (Built-in Plot)

Uses the built-in `plot_surrogate_null_results` method for a quick overview.

In [ ]:
fig = RQA2_ml.plot_surrogate_null_results(
    surr_results,
    save_path=os.path.join(FIG_DIR, "fig2_surrogate_null_overview.png"),
    title="Surrogate Null Hypothesis Testing (Supervised)",
)
plt.show()

### Figure 3: Publication-Quality Surrogate Null Figure

Custom multi-panel figure with:
- **(A)** Accuracy: real nested CV distribution vs surrogate null distributions
- **(B)** ROC AUC: same comparison
- Significance stars and effect sizes annotated

In [ ]:
def fig3_surrogate_null_publication(results, save_path=None):
    """Publication-quality surrogate null figure."""
    real = results["real"]
    surr = results["surrogates"]
    perm = results.get("permutation")
    corrected = results["corrected_p_values"]

    real_acc = real["accuracy"]
    real_auc = real["roc_auc"]

    # Collect data
    null_names = list(surr.keys())
    if perm is not None:
        null_names.append("Permutation")

    palette = sns.color_palette("Set2", len(null_names))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

    for ax, metric, real_scores, ylabel in [
        (axes[0], "accuracy", real_acc, "Classification Accuracy"),
        (axes[1], "roc_auc", real_auc, "ROC AUC"),
    ]:
        # Prepare data for grouped box/violin
        plot_data = []
        plot_labels = []

        # Real data
        for v in real_scores:
            plot_data.append({"Null Type": "Real", "Score": v})

        for kind in surr:
            null_key = f"null_{metric.replace('accuracy','accuracies').replace('roc_auc','roc_aucs')}"
            for v in surr[kind][null_key]:
                plot_data.append({"Null Type": kind, "Score": v})

        if perm is not None:
            perm_key = f"null_{metric}"
            for v in perm[perm_key]:
                plot_data.append({"Null Type": "Permutation", "Score": v})

        df_plot = pd.DataFrame(plot_data)
        order = ["Real"] + list(surr.keys())
        if perm is not None:
            order.append("Permutation")

        # Color mapping
        color_map = {"Real": "#C44E52"}
        for i, k in enumerate(list(surr.keys()) + (["Permutation"] if perm else [])):
            color_map[k] = palette[i]

        bp = sns.boxplot(
            data=df_plot, x="Null Type", y="Score", order=order,
            palette=color_map, width=0.5, linewidth=0.8,
            fliersize=2, ax=ax,
        )

        ax.axhline(1.0 / len(np.unique(y)), color="grey", linestyle=":",
                    linewidth=0.8, label="Chance level")
        ax.set_ylabel(ylabel)
        ax.set_xlabel("")
        ax.legend(fontsize=7, loc="lower left")
        sns.despine(ax=ax)

        # Annotate p-values above each null box
        adj_p = corrected[metric]["adjusted_p"]
        ylim = ax.get_ylim()
        y_top = ylim[1]
        for i, kind in enumerate(list(surr.keys()) + (["Permutation"] if perm else [])):
            key = kind.lower() if kind != "Permutation" else "permutation"
            p = adj_p.get(kind, adj_p.get(key, np.nan))
            if np.isnan(p):
                continue
            stars = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "n.s."))
            ax.text(i + 1, y_top - 0.02 * (ylim[1] - ylim[0]),
                    f"p={p:.3f} {stars}", ha="center", va="top", fontsize=7,
                    fontstyle="italic")

    fig.suptitle(
        "Figure 3: Surrogate-Based Null Hypothesis Testing for Classification",
        fontsize=11, y=1.02,
    )
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path + ".pdf")
        fig.savefig(save_path + ".png")
        print(f"Saved {save_path}.pdf / .png")
    return fig

fig = fig3_surrogate_null_publication(
    surr_results,
    save_path=os.path.join(FIG_DIR, "fig3_surrogate_null_supervised"),
)
plt.show()

### Figure 4: Effect Size Comparison Across Surrogate Types

Bar chart of Cohen's d for each surrogate type, showing *how strongly* real performance exceeds each null — not just significance.

In [ ]:
summary = surr_results["summary"]

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

for ax, metric, label in [
    (axes[0], "effect_size_accuracy", "Cohen's d (Accuracy)"),
    (axes[1], "effect_size_roc_auc", "Cohen's d (ROC AUC)"),
]:
    vals = summary[metric].values
    names = summary["null_type"].values
    colors = ["#C44E52" if v > 0.8 else "#DD8452" if v > 0.5 else "#CCCCCC"
              for v in vals]
    bars = ax.barh(names, vals, color=colors, edgecolor="white", height=0.5)
    ax.set_xlabel(label)
    ax.axvline(0.8, color="grey", linestyle="--", linewidth=0.7, label="Large effect (0.8)")
    ax.axvline(0.5, color="grey", linestyle=":", linewidth=0.7, label="Medium effect (0.5)")
    ax.legend(fontsize=7)
    sns.despine(ax=ax)

fig.suptitle("Figure 4: Effect Sizes — Real vs Surrogate Null Performance", fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig4_effect_sizes.pdf"))
fig.savefig(os.path.join(FIG_DIR, "fig4_effect_sizes.png"))
plt.show()

### Figure 5: Feature Importance (Permutation-Based)

Which RQA features contribute most to classification? This helps interpret *what* nonlinear properties differ between the chaotic systems.

In [ ]:
# Refit best model on all data for feature importance
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

est = ml._make_model(BEST_MODEL, random_state=SEED)
pipe = make_pipeline(StandardScaler(), est)
pipe.fit(X, y)

importance_df = RQA2_ml.feature_importance(
    pipe, X, y, n_repeats=20, random_state=SEED
)
# Assign feature names
importance_df["feature"] = feature_cols

fig = RQA2_ml.plot_feature_importance(
    importance_df, top_n=15,
    save_path=os.path.join(FIG_DIR, "fig5_feature_importance.png"),
    title="Figure 5: Permutation Feature Importance (Top 15)",
)
fig.savefig(os.path.join(FIG_DIR, "fig5_feature_importance.pdf"))
plt.show()

print("\nTop 10 features:")
print(importance_df.head(10).to_string(index=False))

### Figure 6: Feature Selection Frequency

How often was each feature selected across the 100 outer iterations of nested CV? High frequency indicates consistent discriminative value.

In [ ]:
freq = surr_results["real"]["feature_frequency"]
top_freq = freq.head(15).iloc[::-1]

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(top_freq.index, top_freq.values, color="#4C72B0", edgecolor="white")
ax.set_xlabel("Selection frequency (out of 100 outer iterations)")
ax.set_title("Figure 6: Feature Selection Frequency (Nested CV)")
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig6_feature_selection_freq.pdf"))
fig.savefig(os.path.join(FIG_DIR, "fig6_feature_selection_freq.png"))
plt.show()

## 5. Unsupervised: Surrogate Cluster Validation

Can clustering algorithms discover the three dynamical systems *without labels*? And critically, is the clustering quality driven by nonlinear dynamics or merely by linear spectral properties?

In [ ]:
# Real-data clustering with ground-truth comparison
cluster_df, cluster_labels = ml.unsupervised_benchmark(
    X, y_true=y, methods=("kmeans", "gmm", "agglo"),
    k_range=(2, 5), scaler=True, random_state=SEED,
)
print("Clustering validity indices:")
print(cluster_df.to_string(index=False))

In [ ]:
%%time

# Surrogate cluster validation
surr_cluster_results = ml.surrogate_cluster_validation(
    signals,
    window_size=WINDOW_SIZE,
    window_step=WINDOW_STEP,
    window_stats=("mean", "median", "mode"),
    include_params=True,
    surrogate_kinds=("FT", "AAFT", "IAAFT"),
    n_surrogate_iterations=N_SURR_ITER,
    methods=("kmeans", "gmm", "agglo"),
    k_range=(2, 5),
    scaler=True,
    random_state=SEED,
    alpha=0.05,
    correction="fdr_bh",
    verbose=True,
)

print("\nDone.")

### Table 2: Unsupervised Surrogate Null Summary

In [ ]:
cluster_summary = surr_cluster_results["summary"]

# Show silhouette results
sil_cols = [
    "surrogate", "cluster_method",
    "real_silhouette", "null_mean_silhouette", "null_std_silhouette",
    "p_value_silhouette", "adjusted_p_silhouette", "significant_silhouette",
    "effect_size_silhouette",
]
table2 = cluster_summary[sil_cols].copy()
table2.columns = [
    "Surrogate", "Method",
    "Real Sil.", "Null Mean Sil.", "Null Std",
    "p (raw)", "p (FDR)", "Sig.", "Cohen's d",
]
for col in table2.select_dtypes(include="number").columns:
    table2[col] = table2[col].map(lambda x: f"{x:.4f}" if not np.isnan(x) and abs(x) < 100 else f"{x:.1f}")
table2

In [ ]:
# Save full cluster summary
cluster_summary.to_csv(
    os.path.join(FIG_DIR, "table2_surrogate_null_unsupervised.csv"),
    index=False,
)
print("Saved table2_surrogate_null_unsupervised.csv")

### Figure 7: Surrogate Cluster Validation (Silhouette)

In [ ]:
fig = RQA2_ml.plot_surrogate_cluster_validation(
    surr_cluster_results,
    metric="silhouette",
    save_path=os.path.join(FIG_DIR, "fig7_surrogate_cluster_silhouette.png"),
    title="Figure 7: Surrogate Null — Clustering Silhouette",
)
fig.savefig(os.path.join(FIG_DIR, "fig7_surrogate_cluster_silhouette.pdf"))
plt.show()

### Figure 8: Publication-Quality Clustering Null (All Validity Indices)

In [ ]:
def fig8_cluster_null_publication(results, save_path=None):
    """Publication figure: all validity indices for best clustering method."""
    summary = results["summary"]
    surr_data = results["surrogates"]
    real_best = results["real"]["best_per_method"]
    metrics = ["silhouette", "calinski_harabasz", "davies_bouldin"]
    metric_labels = ["Silhouette", "Calinski-Harabasz", "Davies-Bouldin"]

    # Focus on kmeans (most commonly reported)
    method = "kmeans"
    surrogate_kinds = summary["surrogate"].unique()

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    for ax, metric, mlabel in zip(axes, metrics, metric_labels):
        plot_data = []

        # Real value (single point, not a distribution — show as hline)
        real_val = real_best.get(method, {}).get(metric, np.nan)

        for kind in surrogate_kinds:
            mr = surr_data.get(kind, {}).get(method, {})
            null_arr = mr.get(f"null_{metric}", np.array([]))
            for v in null_arr[~np.isnan(null_arr)]:
                plot_data.append({"Surrogate": kind, "Score": v})

        if not plot_data:
            ax.text(0.5, 0.5, "No data", transform=ax.transAxes, ha="center")
            continue

        df_plot = pd.DataFrame(plot_data)
        palette = sns.color_palette("Set2", len(surrogate_kinds))
        color_map = {k: palette[i] for i, k in enumerate(surrogate_kinds)}

        sns.boxplot(data=df_plot, x="Surrogate", y="Score",
                    palette=color_map, width=0.5, linewidth=0.8,
                    fliersize=2, ax=ax)

        if not np.isnan(real_val):
            ax.axhline(real_val, color="#C44E52", linewidth=2,
                       linestyle="--", label=f"Real ({real_val:.3f})")

        ax.set_ylabel(mlabel)
        ax.set_xlabel("")
        ax.legend(fontsize=7, loc="best")
        sns.despine(ax=ax)

        # Annotate p-values
        for i, kind in enumerate(surrogate_kinds):
            mr = surr_data.get(kind, {}).get(method, {})
            p = mr.get(f"p_value_{metric}", np.nan)
            if np.isnan(p):
                continue
            stars = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "n.s."))
            ylim = ax.get_ylim()
            ax.text(i, ylim[1] - 0.03 * (ylim[1] - ylim[0]),
                    f"p={p:.3f}\n{stars}", ha="center", va="top", fontsize=7)

    fig.suptitle(
        f"Figure 8: Surrogate Null Testing for Clustering Validity (K-Means)",
        fontsize=11, y=1.02,
    )
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path + ".pdf")
        fig.savefig(save_path + ".png")
        print(f"Saved {save_path}.pdf / .png")
    return fig

fig = fig8_cluster_null_publication(
    surr_cluster_results,
    save_path=os.path.join(FIG_DIR, "fig8_cluster_null_all_indices"),
)
plt.show()

### Figure 9: PCA Cluster Scatter (Real vs Ground Truth)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

# (A) Ground-truth labels
from sklearn.decomposition import PCA
X_scaled = StandardScaler().fit_transform(X)
X_2d = PCA(n_components=2).fit_transform(X_scaled)

for ax, label_set, title_str in [
    (axes[0], y, "(A) Ground Truth"),
    (axes[1], cluster_labels.get("kmeans", y), "(B) K-Means Clustering"),
]:
    for lab in np.unique(label_set):
        mask = np.asarray(label_set) == lab
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   label=str(lab), alpha=0.7, s=40,
                   edgecolors="white", linewidth=0.3)
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    ax.legend(fontsize=8)
    ax.set_title(title_str)
    sns.despine(ax=ax)

fig.suptitle("Figure 9: PCA Projection of RQA Features", fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig9_pca_scatter.pdf"))
fig.savefig(os.path.join(FIG_DIR, "fig9_pca_scatter.png"))
plt.show()

## 6. Cluster Stability Assessment

In [ ]:
stability = ml.cluster_stability(
    X, method="kmeans", n_clusters=3,
    n_bootstrap=100, subsample_fraction=0.8,
    scaler=True, random_state=SEED,
)
print(f"K-Means cluster stability (k=3):")
print(f"  Mean ARI: {stability['mean_ari']:.3f} +/- {stability['std_ari']:.3f}")

stability_gmm = ml.cluster_stability(
    X, method="gmm", n_clusters=3,
    n_bootstrap=100, subsample_fraction=0.8,
    scaler=True, random_state=SEED,
)
print(f"\nGMM cluster stability (k=3):")
print(f"  Mean ARI: {stability_gmm['mean_ari']:.3f} +/- {stability_gmm['std_ari']:.3f}")

## 7. Summary: All Outputs

### Figures saved to `figures/`
| File | Description |
|------|-------------|
| `fig1_example_timeseries` | Example signals from three chaotic systems |
| `fig2_surrogate_null_overview` | Built-in surrogate null violin plot |
| `fig3_surrogate_null_supervised` | Publication-quality supervised null (boxplots + p-values) |
| `fig4_effect_sizes` | Cohen's d bar chart across surrogate types |
| `fig5_feature_importance` | Permutation feature importance |
| `fig6_feature_selection_freq` | Nested CV feature selection frequency |
| `fig7_surrogate_cluster_silhouette` | Built-in cluster validation plot |
| `fig8_cluster_null_all_indices` | All three validity indices vs surrogate null |
| `fig9_pca_scatter` | PCA projection: ground truth vs clustering |

### Tables saved to `figures/`
| File | Description |
|------|-------------|
| `table1_surrogate_null_supervised.csv` | Full supervised surrogate null results |
| `table2_surrogate_null_unsupervised.csv` | Full unsupervised surrogate null results |

In [ ]:
print("All figures and tables generated.")
print(f"\nFigure directory: {os.path.abspath(FIG_DIR)}")
print(f"Files: {sorted(os.listdir(FIG_DIR))}")